In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from google.colab import files
import io
import matplotlib.patches as mpatches

# 1. Upload dos arquivos
print("Selecione os arquivos CSV (mqttsn e p4ssn):")
uploaded = files.upload()

# 2. Carregamento e Tratamento
dfs = []
for filename in uploaded.keys():
    df_temp = pd.read_csv(io.BytesIO(uploaded[filename]))
    df_temp['cenario'] = df_temp['cenario'].str.upper()
    dfs.append(df_temp)

df_completo = pd.concat(dfs, ignore_index=True)

def format_qos(row):
    label = f"QoS {int(row['qos'])}"
    if row['qos'] == 3: label = "QoS -1"
    if row['qos'] == 0:
        # Alterado para Português conforme solicitado
        label += " (Com Retain)" if row['retain'] == 1 else " (Sem Retain)"
    return label

df_completo['QoS_Config'] = df_completo.apply(format_qos, axis=1)

# Médias para o gráfico
df_plot = df_completo.groupby(['cenario', 'QoS_Config'])['t_total_ms'].mean().reset_index()
df_plot.rename(columns={'cenario': 'Topologia', 't_total_ms': 'Tempo Total Médio (ms)'}, inplace=True)

# 3. Configuração Visual
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 7))

# Ordem do eixo X atualizada com os novos nomes
ordem_x = ["QoS -1", "QoS 0 (Sem Retain)", "QoS 0 (Com Retain)", "QoS 1", "QoS 2"]
palette_colors = sns.color_palette("viridis", 2)

ax = sns.barplot(
    data=df_plot,
    x='QoS_Config',
    y='Tempo Total Médio (ms)',
    hue='Topologia',
    palette=palette_colors,
    order=ordem_x
)

# 4. Aplicação de Hachuras e Legenda Customizada
patterns = ['...', '///']
topologias = sorted(df_plot['Topologia'].unique())

num_bars_per_topo = len(ordem_x)
for i, bar in enumerate(ax.patches):
    if i < num_bars_per_topo:
        bar.set_hatch(patterns[0])
    else:
        bar.set_hatch(patterns[1])

# Legenda que reflete as cores e hachuras corretamente
patch1 = mpatches.Patch(facecolor=palette_colors[0], hatch=patterns[0], label=topologias[0])
patch2 = mpatches.Patch(facecolor=palette_colors[1], hatch=patterns[1], label=topologias[1])

plt.legend(handles=[patch1, patch2], title='Topologia', loc='upper left', frameon=True)

# Finalização
plt.title('Compatibilidade Funcional: P4SSN x MQTT-SN', fontsize=15, pad=20)
plt.xlabel('Nível de QoS', fontsize=12)
plt.ylabel('Tempo Total Médio do Fluxo (ms)', fontsize=12)

plt.savefig("grafico_compatibilidade_funcional.eps", format='eps', bbox_inches='tight')
plt.show()